# Logic Circuit Classification with Qwen3-VL-8B, QLoRA, and Adaptive Multiview Inference

This notebook fine-tunes **Qwen3-VL-8B-Instruct** for visual logic-circuit classification using **4-bit QLoRA** and evaluates an adaptive multiview inference strategy.

The pipeline includes:

1. Dataset loading, stratified train/validation splitting, and training-set balancing.
2. Supervised multimodal fine-tuning with constrained answer formats.
3. Next-token scoring over valid labels (`0`–`7` or `True`/`False`).
4. Image augmentation with contrast enhancement and whitespace cropping.
5. Aggregation of class log-probabilities across views.
6. Calibration and holdout evaluation to determine whether adaptive multiview inference improves over single-view inference.
7. Test-set prediction and CSV export.

The inference procedure combines **test-time augmentation** with probability-based ensembling. It does not rely on external OCR or additional models.

**Environment:** Kaggle notebook with an NVIDIA T4 GPU and internet access. Dataset files must be supplied separately.


## 1. Dependencies

Enable a **T4 GPU** and internet access in Kaggle before running the installation cells. Restart the notebook session if required by the package installer.


In [ ]:
%pip install -q -U "ms-swift>=4.0,<5" "transformers>=4.57,<5" "datasets>=3,<4.8.5" "peft>=0.17,<0.20" "accelerate>=1.2" "bitsandbytes>=0.45" sentencepiece


In [ ]:
%pip install -q -U "qwen-vl-utils>=0.0.14"

## 2. Configuration and reproducibility

Configuration values are optimized for a memory-constrained GPU. Adjust training epochs, image resolution, and batch size based on available resources and validation results.


In [ ]:
import gc
import json
import os
import random
import re
import subprocess
import sys
from collections import Counter
from pathlib import Path

from PIL import Image, ImageEnhance, ImageOps

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"
DATASET_DIR = None  # Optional: Path("/kaggle/input/dataset-name")

WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./output")
DATA_DIR = WORK_DIR / "circuit_qlora_data"
TRAIN_OUTPUT_DIR = WORK_DIR / "qwen3vl8b_circuit_qlora"
SUBMISSION_PATH = WORK_DIR / "submission_qwen8b_probability_ttc.csv"
DATA_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Training
NUM_EPOCHS = 1
OUTPUT_TRAIN_CAP = 450   # Limit binary-output examples to balance training tasks
MAX_PIXELS = 768 * 28 * 28
MIN_PIXELS = 256 * 28 * 28

# Adaptive multiview probability ensemble
# Thresholds are calibrated on validation data
COUNT_CANDIDATES = [str(value) for value in range(8)]
OUTPUT_CANDIDATES = ["True", "False"]
TTC_THRESHOLD_GRID = np.linspace(0.0, 0.90, 10)
TTC_METHOD_VERSION = "probability_multiview_v1"

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["MAX_PIXELS"] = str(MAX_PIXELS)

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))


## 3. Dataset loading

The notebook searches for the expected JSONL files under `/kaggle/input` and the current directory. Set `DATASET_DIR` explicitly if multiple datasets contain files with the same names.


In [ ]:
def find_unique_file(filename: str) -> Path:
    roots = [Path("/kaggle/input"), Path(".")]
    matches = []
    for root in roots:
        if root.exists():
            matches.extend(root.rglob(filename))
    matches = sorted(set(p.resolve() for p in matches))
    if not matches:
        raise FileNotFoundError(f"Não encontrei {filename}. Defina DATASET_DIR manualmente.")
    if len(matches) > 1:
        print(f"Aviso: {len(matches)} cópias de {filename}; usando {matches[0]}")
    return matches[0]


if DATASET_DIR is None:
    train_questions_path = find_unique_file("questions_with_answers.jsonl")
    test_questions_path = find_unique_file("questions.jsonl")
    train_root = train_questions_path.parent.parent
    test_root = test_questions_path.parent.parent
else:
    dataset_dir = Path(DATASET_DIR)
    train_root = dataset_dir / "train_dataset"
    test_root = dataset_dir / "test_dataset"
    train_questions_path = train_root / "text" / "questions_with_answers.jsonl"
    test_questions_path = test_root / "text" / "questions.jsonl"


def read_jsonl(path: Path) -> pd.DataFrame:
    with path.open("r", encoding="utf-8") as f:
        return pd.DataFrame(json.loads(line) for line in f if line.strip())


def task_from_question(question: str) -> str:
    question = str(question).strip()
    if question.lower().startswith("what is the output"):
        return "output"
    match = re.search(
        r"how many\s+(xnor|xor|nand|nor|and|or|not)\s+gates",
        question,
        flags=re.IGNORECASE,
    )
    if not match:
        raise ValueError(f"Pergunta desconhecida: {question}")
    return match.group(1).lower()


def get_image_path(root: Path, index: int) -> Path:
    path = root / "images" / f"circuit_{int(index)}.png"
    if not path.exists():
        raise FileNotFoundError(path)
    return path.resolve()


train_df = read_jsonl(train_questions_path)
test_df = read_jsonl(test_questions_path)
train_df["answer"] = train_df["answer"].astype(str)
train_df["task"] = train_df["question"].map(task_from_question)
test_df["task"] = test_df["question"].map(task_from_question)

assert len(train_df) == 2000, f"Esperava 2.000 exemplos; encontrei {len(train_df)}"
assert len(test_df) == 1000, f"Esperava 1.000 exemplos; encontrei {len(test_df)}"
assert train_df["index"].is_unique and test_df["index"].is_unique

print("Treino:", train_questions_path)
print("Teste:", test_questions_path)
display(pd.DataFrame({
    "treino": train_df["task"].value_counts(),
    "teste": test_df["task"].value_counts(),
}).fillna(0).astype(int).sort_index())


## 4. Train/validation split and sampling

Validation data preserves the original label distribution and is excluded from fine-tuning. Training uses all gate-counting examples and caps the number of binary-output examples to reduce class-task imbalance.


In [ ]:
# Stratify by task and label; group singleton combinations in a rare stratum
raw_strata = train_df["task"] + "::" + train_df["answer"]
freq = raw_strata.value_counts()
strata = raw_strata.where(raw_strata.map(freq) >= 2, "__rare__")

fit_df, val_df = train_test_split(
    train_df,
    test_size=0.10,
    random_state=SEED,
    stratify=strata,
)

fit_output = fit_df[fit_df["task"] == "output"]
fit_count = fit_df[fit_df["task"] != "output"]
if len(fit_output) > OUTPUT_TRAIN_CAP:
    fit_output = fit_output.sample(OUTPUT_TRAIN_CAP, random_state=SEED)

fit_balanced_df = (
    pd.concat([fit_count, fit_output], ignore_index=True)
    .sample(frac=1, random_state=SEED)
    .reset_index(drop=True)
)
val_df = val_df.reset_index(drop=True)

assert set(fit_balanced_df["index"]).isdisjoint(set(val_df["index"]))
print("Treino original após split:", len(fit_df))
print("Treino balanceado:", len(fit_balanced_df))
print("Validação intocada:", len(val_df))
display(pd.DataFrame({
    "ajuste": fit_balanced_df["task"].value_counts(),
    "validacao": val_df["task"].value_counts(),
}).fillna(0).astype(int).sort_index())


## 5. Supervised multimodal dataset format

Each training example includes a circuit image, a question, and the corresponding target label. The supervised response contains only the answer, without generated reasoning traces.


In [ ]:
SYSTEM_PROMPT = (
    "You solve classification questions about logic-circuit diagrams. "
    "Return exactly one answer and no explanation: an integer for gate-counting "
    "questions, or True/False for output questions."
)


def to_swift_record(row, root: Path, include_answer: bool = True):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": "<image>\n" + str(row["question"]).strip() + "\nAnswer only with the exact label.",
        },
    ]
    if include_answer:
        messages.append({"role": "assistant", "content": str(row["answer"]).strip()})
    return {
        "messages": messages,
        "images": [str(get_image_path(root, int(row["index"])))],
    }


def write_swift_jsonl(df: pd.DataFrame, root: Path, path: Path):
    with path.open("w", encoding="utf-8") as out:
        for _, row in df.iterrows():
            out.write(json.dumps(to_swift_record(row, root), ensure_ascii=False) + "\n")


swift_train_path = DATA_DIR / "train_multimodal.jsonl"
swift_val_path = DATA_DIR / "val_multimodal.jsonl"
write_swift_jsonl(fit_balanced_df, train_root, swift_train_path)
write_swift_jsonl(val_df, train_root, swift_val_path)

print("Arquivos preparados:")
print(swift_train_path, "->", sum(1 for _ in swift_train_path.open()))
print(swift_val_path, "->", sum(1 for _ in swift_val_path.open()))
print("Exemplo (caminho abreviado):")
example = json.loads(swift_train_path.open(encoding="utf-8").readline())
example["images"] = [Path(example["images"][0]).name]
display(example)


## 6. QLoRA fine-tuning

- **4-bit quantization (NF4):** reduces memory requirements for the base model.
- **LoRA (rank 16):** trains lightweight adapters instead of updating the full model weights.
- **Frozen visual encoder:** limits training memory consumption.
- **Trainable alignment and language components:** enable adaptation to circuit classification.
- **FP16 and SDPA:** support execution on NVIDIA T4 GPUs without requiring FlashAttention.

Training checkpoints are written to the configured output directory. Set `RUN_TRAINING=False` to reuse existing checkpoints.


In [ ]:
%pip uninstall -q -y torchao

In [ ]:
import os
import subprocess
from pathlib import Path

RUN_TRAINING = True

LOG_PATH = WORK_DIR / "swift_train.log"

train_command = [
    "swift",
    "sft",

    # Model and data
    "--model", MODEL_ID,
    "--use_hf", "true",
    "--dataset", str(swift_train_path),
    "--val_dataset", str(swift_val_path),

    # 4-bit QLoRA
    "--tuner_type", "lora",
    "--quant_method", "bnb",
    "--quant_bits", "4",
    "--bnb_4bit_compute_dtype", "float16",
    "--torch_dtype", "float16",
    "--attn_impl", "sdpa",

    # Trainable components
    "--freeze_vit", "true",
    "--freeze_aligner", "false",
    "--freeze_llm", "false",
    "--target_modules", "all-linear",

    # LoRA configuration
    "--lora_rank", "16",
    "--lora_alpha", "32",
    "--lora_dropout", "0.05",

    # Training
    "--num_train_epochs", str(NUM_EPOCHS),
    "--per_device_train_batch_size", "1",
    "--per_device_eval_batch_size", "1",
    "--gradient_accumulation_steps", "12",
    "--gradient_checkpointing", "true",
    "--learning_rate", "0.0001",
    "--warmup_ratio", "0.05",
    "--weight_decay", "0.01",

    # Input resolution
    "--max_length", "1152",
    "--loss_scale", "last_round",

    # Validation and checkpoints
    "--eval_steps", "50",
    "--save_steps", "50",
    "--logging_steps", "5",
    "--save_total_limit", "2",
    "--load_best_model_at_end", "true",
    "--metric_for_best_model", "eval_loss",
    "--greater_is_better", "false",

    # Reproducibility and output
    "--dataloader_num_workers", "2",
    "--seed", str(SEED),
    "--data_seed", str(SEED),
    "--report_to", "none",
    "--output_dir", str(TRAIN_OUTPUT_DIR),
]

print("Comando de treinamento:")
print(" ".join(train_command))
print("\nLog:", LOG_PATH)

if RUN_TRAINING:
    if not torch.cuda.is_available():
        raise RuntimeError("Ative a GPU T4 no Kaggle antes do treinamento.")

    # Image resolution limit for the multimodal processor
    training_environment = os.environ.copy()
    training_environment["MAX_PIXELS"] = str(MAX_PIXELS)
    training_environment["TOKENIZERS_PARALLELISM"] = "false"
    training_environment["PYTORCH_CUDA_ALLOC_CONF"] = (
        "expandable_segments:True"
    )

    last_lines = []

    with LOG_PATH.open("w", encoding="utf-8") as log_file:
        process = subprocess.Popen(
            train_command,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env=training_environment,
        )

        for line in process.stdout:
            print(line, end="")

            log_file.write(line)
            log_file.flush()

            last_lines.append(line)

            if len(last_lines) > 150:
                last_lines.pop(0)

        return_code = process.wait()

    if return_code != 0:
        print("\n" + "=" * 80)
        print("CAUSA REAL DO ERRO")
        print("=" * 80)
        print("".join(last_lines))

        raise RuntimeError(
            f"O swift encerrou com código {return_code}. "
            f"Log completo em: {LOG_PATH}"
        )

    print("\nTreinamento concluído com sucesso.")
    print("Checkpoints salvos em:", TRAIN_OUTPUT_DIR)

else:
    print("Treinamento ignorado.")
    print("Será utilizado um checkpoint existente em:", TRAIN_OUTPUT_DIR)

## 7. Checkpoint selection and model loading

Load the quantized base model and attach the selected LoRA adapter for inference.


In [ ]:
def checkpoint_step(path: Path) -> int:
    match = re.search(r"checkpoint-(\d+)$", path.name)
    return int(match.group(1)) if match else -1


def find_best_adapter(output_dir: Path) -> Path:
    states = sorted(
        output_dir.rglob("trainer_state.json"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    for state_path in states:
        try:
            state = json.loads(state_path.read_text())
            best = state.get("best_model_checkpoint")
            if best and Path(best).exists():
                return Path(best)
        except Exception:
            pass
    checkpoints = [p for p in output_dir.rglob("checkpoint-*") if p.is_dir()]
    if not checkpoints:
        raise FileNotFoundError("Nenhum checkpoint encontrado. Execute a célula de treino.")
    return max(checkpoints, key=lambda p: p.stat().st_mtime)


ADAPTER_DIR = find_best_adapter(TRAIN_OUTPUT_DIR)
print("Adaptador selecionado:", ADAPTER_DIR)
print("Arquivos:", sorted(p.name for p in ADAPTER_DIR.iterdir())[:12])


In [ ]:
from peft import PeftModel
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3VLForConditionalGeneration

gc.collect()
torch.cuda.empty_cache()

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
)
processor.tokenizer.padding_side = "left"
if processor.tokenizer.pad_token_id is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

base_model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    attn_implementation="sdpa",
    low_cpu_mem_usage=True,
)
model = PeftModel.from_pretrained(base_model, str(ADAPTER_DIR))
model.eval()
model.generation_config.pad_token_id = processor.tokenizer.pad_token_id

print("Modelo QLoRA carregado.")
print("Dispositivo:", next(model.parameters()).device)


## 8. Constrained label scoring

Instead of unrestricted text generation, the model scores the next token for each valid label: `0`–`7` for gate-counting questions and `True`/`False` for circuit-output questions.

Scores are normalized over the candidate labels for each task. The implementation verifies that every label maps to a single tokenizer token before running inference.


In [ ]:
def valid_candidates(task: str) -> list[str]:
    return OUTPUT_CANDIDATES if task == "output" else COUNT_CANDIDATES


def build_messages(question: str, image):
    return [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {
                    "type": "text",
                    "text": str(question).strip() + "\nAnswer only with the exact label.",
                },
            ],
        },
    ]


def candidate_token_table():
    labels = COUNT_CANDIDATES + OUTPUT_CANDIDATES
    table = {}
    for label in labels:
        token_ids = processor.tokenizer.encode(label, add_special_tokens=False)
        if len(token_ids) != 1:
            raise RuntimeError(
                f"O rótulo {label!r} usa {len(token_ids)} tokens ({token_ids}). "
                "Esta implementação de scoring requer um token por rótulo."
            )
        table[label] = int(token_ids[0])
    if len(set(table.values())) != len(table):
        raise RuntimeError(f"Rótulos diferentes compartilharam token: {table}")
    return table


CANDIDATE_TOKEN_ID = candidate_token_table()
print("Tokens dos rótulos:", CANDIDATE_TOKEN_ID)


def trim_white_margins(image: Image.Image, threshold: int = 248, padding: int = 20) -> Image.Image:
    rgb = image.convert("RGB")
    gray = np.asarray(rgb.convert("L"))
    ys, xs = np.where(gray < threshold)
    if len(xs) == 0:
        return rgb
    left = max(0, int(xs.min()) - padding)
    right = min(rgb.width, int(xs.max()) + padding + 1)
    top = max(0, int(ys.min()) - padding)
    bottom = min(rgb.height, int(ys.max()) + padding + 1)
    cropped = rgb.crop((left, top, right, bottom))
    return ImageOps.expand(cropped, border=padding, fill="white")


def make_ttc_views(image_file: Path) -> dict[str, Image.Image]:
    with Image.open(image_file) as opened:
        original = opened.convert("RGB").copy()
    # Enhanced view: emphasize lines and logic-gate details
    enhanced = ImageOps.autocontrast(original.convert("L"), cutoff=0.5).convert("RGB")
    enhanced = ImageEnhance.Sharpness(enhanced).enhance(1.6)
    # Cropped view: remove surrounding whitespace
    cropped = trim_white_margins(original)
    return {"original": original, "enhanced": enhanced, "cropped": cropped}


def move_inputs_to_model(inputs):
    device = model.get_input_embeddings().weight.device
    moved = {}
    for key, value in inputs.items():
        moved[key] = value.to(device) if hasattr(value, "to") else value
    return moved


@torch.inference_mode()
def score_valid_labels(question: str, image, task: str) -> dict:
    messages = build_messages(question, image)
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    )
    inputs = move_inputs_to_model(inputs)
    outputs = model(**inputs, use_cache=False, return_dict=True)
    next_token_logits = outputs.logits[0, -1].float()

    candidates = valid_candidates(task)
    ids = torch.tensor(
        [CANDIDATE_TOKEN_ID[label] for label in candidates],
        device=next_token_logits.device,
        dtype=torch.long,
    )
    candidate_logits = next_token_logits.index_select(0, ids)
    log_probs_tensor = torch.log_softmax(candidate_logits, dim=0)
    probs_tensor = torch.softmax(candidate_logits, dim=0)
    order = torch.argsort(probs_tensor, descending=True)
    best_position = int(order[0])
    margin = float(probs_tensor[order[0]] - probs_tensor[order[1]])

    result = {
        "answer": candidates[best_position],
        "margin": margin,
        "log_probs": {
            label: float(log_probs_tensor[position])
            for position, label in enumerate(candidates)
        },
    }
    del inputs, outputs, next_token_logits, candidate_logits
    return result


def combine_log_scores(score_maps: list[dict], weights=None) -> dict:
    if not score_maps:
        raise ValueError("É necessária pelo menos uma visão.")
    candidates = list(score_maps[0])
    if weights is None:
        weights = [1.0] * len(score_maps)
    weights = np.asarray(weights, dtype=np.float64)
    weights = weights / weights.sum()
    combined = np.zeros(len(candidates), dtype=np.float64)
    for weight, scores in zip(weights, score_maps):
        if list(scores) != candidates:
            raise ValueError("As visões possuem candidatos em ordens diferentes.")
        combined += weight * np.asarray([scores[label] for label in candidates])
    probability = np.exp(combined - combined.max())
    probability /= probability.sum()
    order = np.argsort(-probability)
    return {
        "answer": candidates[int(order[0])],
        "margin": float(probability[order[0]] - probability[order[1]]),
        "log_probs": {label: float(value) for label, value in zip(candidates, combined)},
    }


def direct_prediction(question: str, image_file: Path, task: str) -> dict:
    image = make_ttc_views(image_file)["original"]
    return score_valid_labels(question, image, task)


## 9. Multiview validation cache

Compute and cache class log-probabilities for the original, enhanced, and cropped image views. Cached results support comparison of single-view, two-view, three-view, and adaptive inference policies without repeating forward passes.


In [ ]:
adapter_tag = re.sub(r"[^A-Za-z0-9_-]+", "_", f"{ADAPTER_DIR.parent.name}_{ADAPTER_DIR.name}")
VAL_SCORE_CACHE = WORK_DIR / f"validation_scores_{adapter_tag}_{TTC_METHOD_VERSION}.jsonl"


def load_jsonl_by_index(path: Path) -> dict[int, dict]:
    records = {}
    if path.exists():
        with path.open("r", encoding="utf-8") as stream:
            for line in stream:
                if line.strip():
                    item = json.loads(line)
                    records[int(item["index"])] = item
    return records


validation_cache = load_jsonl_by_index(VAL_SCORE_CACHE)
print("Cache de validação:", VAL_SCORE_CACHE)
print("Exemplos já processados:", len(validation_cache))

with VAL_SCORE_CACHE.open("a", encoding="utf-8") as out:
    for _, row in tqdm(val_df.iterrows(), total=len(val_df), desc="Scoring multiview"):
        idx = int(row["index"])
        if idx in validation_cache:
            continue
        views = make_ttc_views(get_image_path(train_root, idx))
        view_results = {
            name: score_valid_labels(row["question"], image, row["task"])
            for name, image in views.items()
        }
        item = {
            "index": idx,
            "task": row["task"],
            "answer": str(row["answer"]),
            "views": view_results,
        }
        out.write(json.dumps(item, ensure_ascii=False) + "\n")
        out.flush()
        validation_cache[idx] = item

val_score_records = [validation_cache[int(idx)] for idx in val_df["index"]]
val_scores = pd.DataFrame(val_score_records)

val_scores["prediction_original"] = val_scores["views"].map(
    lambda item: item["original"]["answer"]
)
val_scores["prediction_two_views"] = val_scores["views"].map(
    lambda item: combine_log_scores([
        item["original"]["log_probs"], item["enhanced"]["log_probs"]
    ])["answer"]
)
val_scores["prediction_three_views"] = val_scores["views"].map(
    lambda item: combine_log_scores([
        item["original"]["log_probs"],
        item["enhanced"]["log_probs"],
        item["cropped"]["log_probs"],
    ])["answer"]
)
for method in ["original", "two_views", "three_views"]:
    val_scores[f"correct_{method}"] = (
        val_scores["answer"].astype(str) == val_scores[f"prediction_{method}"].astype(str)
    )

display(pd.DataFrame({
    "overall": val_scores[["correct_original", "correct_two_views", "correct_three_views"]].mean(),
    "count": val_scores[val_scores["task"] != "output"][["correct_original", "correct_two_views", "correct_three_views"]].mean(),
    "output": val_scores[val_scores["task"] == "output"][["correct_original", "correct_two_views", "correct_three_views"]].mean(),
}))


## 10. Adaptive inference calibration

Partition the validation set into calibration and holdout subsets. Calibrate uncertainty-margin thresholds on the first subset, then evaluate the selected policy on the holdout subset.

The adaptive policy starts with the original image and processes enhanced or cropped views only when the classification margin is below the corresponding threshold. When accuracy is tied, prefer the policy with fewer average forward passes.


In [ ]:
def apply_adaptive_policy(view_results: dict, threshold_1: float, threshold_2: float):
    original = view_results["original"]
    if original["margin"] >= threshold_1:
        return original["answer"], 1, original["margin"]

    combined_two = combine_log_scores([
        original["log_probs"],
        view_results["enhanced"]["log_probs"],
    ])
    if combined_two["margin"] >= threshold_2:
        return combined_two["answer"], 2, combined_two["margin"]

    combined_three = combine_log_scores([
        original["log_probs"],
        view_results["enhanced"]["log_probs"],
        view_results["cropped"]["log_probs"],
    ])
    return combined_three["answer"], 3, combined_three["margin"]


def evaluate_policy(frame: pd.DataFrame, threshold_1: float, threshold_2: float):
    predictions, paths = [], []
    for view_results in frame["views"]:
        answer, used_paths, _ = apply_adaptive_policy(view_results, threshold_1, threshold_2)
        predictions.append(str(answer))
        paths.append(used_paths)
    correct = np.asarray(predictions) == frame["answer"].astype(str).to_numpy()
    return {
        "accuracy": float(correct.mean()),
        "average_views": float(np.mean(paths)),
        "predictions": predictions,
        "paths": paths,
    }


def search_thresholds(frame: pd.DataFrame):
    best = None
    rows = []
    for threshold_1 in TTC_THRESHOLD_GRID:
        for threshold_2 in TTC_THRESHOLD_GRID:
            result = evaluate_policy(frame, float(threshold_1), float(threshold_2))
            row = {
                "threshold_1": float(threshold_1),
                "threshold_2": float(threshold_2),
                "accuracy": result["accuracy"],
                "average_views": result["average_views"],
            }
            rows.append(row)
            key = (row["accuracy"], -row["average_views"])
            if best is None or key > best[0]:
                best = (key, row)
    return best[1], pd.DataFrame(rows)


family = np.where(val_scores["task"] == "output", "output", "count")
calibration_positions, holdout_positions = train_test_split(
    np.arange(len(val_scores)),
    test_size=0.50,
    random_state=SEED,
    stratify=family,
)
calibration_df = val_scores.iloc[calibration_positions].reset_index(drop=True)
holdout_df = val_scores.iloc[holdout_positions].reset_index(drop=True)

best_calibration, threshold_results = search_thresholds(calibration_df)
TTC_THRESHOLD_1 = best_calibration["threshold_1"]
TTC_THRESHOLD_2 = best_calibration["threshold_2"]

holdout_policy = evaluate_policy(holdout_df, TTC_THRESHOLD_1, TTC_THRESHOLD_2)
holdout_original_accuracy = float(holdout_df["correct_original"].mean())
USE_TTC_FINAL = bool(holdout_policy["accuracy"] > holdout_original_accuracy)

print("Limiar após visão original:", TTC_THRESHOLD_1)
print("Limiar após duas visões:", TTC_THRESHOLD_2)
print(f"Calibração TTC: {best_calibration['accuracy']:.4f}")
print(f"Holdout original: {holdout_original_accuracy:.4f}")
print(f"Holdout TTC: {holdout_policy['accuracy']:.4f}")
print(f"Visões médias no holdout: {holdout_policy['average_views']:.2f}")
print("TTC será usado no teste?", USE_TTC_FINAL)

# After holdout evaluation, refine test-time thresholds using validation data.
if USE_TTC_FINAL:
    best_full, _ = search_thresholds(val_scores)
    TTC_THRESHOLD_1 = best_full["threshold_1"]
    TTC_THRESHOLD_2 = best_full["threshold_2"]
    print("Limiares finais com toda a validação:", TTC_THRESHOLD_1, TTC_THRESHOLD_2)

validation_diagnostic_path = WORK_DIR / "validation_qwen8b_probability_ttc.csv"
diagnostic = val_scores.drop(columns=["views"]).copy()
diagnostic.to_csv(validation_diagnostic_path, index=False)
print("Diagnóstico salvo em:", validation_diagnostic_path)


## 11. Test-set inference and checkpointing

Apply the selected inference policy to test examples. Additional image views are processed only when needed. Prediction checkpoints are namespaced by adapter, method version, and calibrated thresholds to support resumable inference.


In [ ]:
run_tag = re.sub(
    r"[^A-Za-z0-9_-]+",
    "_",
    (
        f"{ADAPTER_DIR.parent.name}_{ADAPTER_DIR.name}_{TTC_METHOD_VERSION}_"
        f"use{int(USE_TTC_FINAL)}_t1{TTC_THRESHOLD_1:.2f}_t2{TTC_THRESHOLD_2:.2f}"
    ),
)
TEST_CHECKPOINT_PATH = WORK_DIR / f"test_predictions_{run_tag}.jsonl"


def load_prediction_checkpoint(path: Path):
    rows = {}
    if path.exists():
        with path.open("r", encoding="utf-8") as stream:
            for line in stream:
                if line.strip():
                    item = json.loads(line)
                    rows[int(item["index"])] = item
    return rows


def predict_test_item(row, root: Path):
    idx = int(row["index"])
    views = make_ttc_views(get_image_path(root, idx))
    original = score_valid_labels(row["question"], views["original"], row["task"])
    view_answers = {"original": original["answer"]}

    if not USE_TTC_FINAL or original["margin"] >= TTC_THRESHOLD_1:
        return original["answer"], 1, original["margin"], view_answers

    enhanced = score_valid_labels(row["question"], views["enhanced"], row["task"])
    view_answers["enhanced"] = enhanced["answer"]
    combined_two = combine_log_scores([original["log_probs"], enhanced["log_probs"]])
    if combined_two["margin"] >= TTC_THRESHOLD_2:
        return combined_two["answer"], 2, combined_two["margin"], view_answers

    cropped = score_valid_labels(row["question"], views["cropped"], row["task"])
    view_answers["cropped"] = cropped["answer"]
    combined_three = combine_log_scores([
        original["log_probs"], enhanced["log_probs"], cropped["log_probs"]
    ])
    return combined_three["answer"], 3, combined_three["margin"], view_answers


completed = load_prediction_checkpoint(TEST_CHECKPOINT_PATH)
print("Checkpoint:", TEST_CHECKPOINT_PATH)
print("Já concluídos:", len(completed))

with TEST_CHECKPOINT_PATH.open("a", encoding="utf-8") as out:
    for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Teste Qwen8B TTC"):
        idx = int(row["index"])
        if idx in completed:
            continue

        answer, views_used, margin, view_answers = predict_test_item(row, test_root)
        item = {
            "index": idx,
            "answer": str(answer),
            "task": row["task"],
            "views_used": int(views_used),
            "used_ttc": bool(views_used > 1),
            "margin": float(margin),
            "view_answers": view_answers,
        }
        out.write(json.dumps(item, ensure_ascii=False) + "\n")
        out.flush()
        completed[idx] = item

print("Total concluído:", len(completed))


## 12. Submission export and validation


In [ ]:
prediction_rows = [completed[int(idx)] for idx in test_df["index"]]
submission = pd.DataFrame({
    "index": test_df["index"].astype(int),
    "answer": [str(item["answer"]) for item in prediction_rows],
})

assert list(submission.columns) == ["index", "answer"]
assert len(submission) == 1000
assert submission["index"].is_unique
assert not submission["answer"].isna().any()

for answer, task in zip(submission["answer"], test_df["task"]):
    if task == "output":
        assert answer in {"True", "False"}, (task, answer)
    else:
        assert re.fullmatch(r"\d+", answer), (task, answer)

submission.to_csv(SUBMISSION_PATH, index=False)
print("Submissão salva em:", SUBMISSION_PATH)
print("TTC aplicado em:", sum(bool(item["used_ttc"]) for item in prediction_rows), "amostras")
display(submission.head(10))
display(submission["answer"].value_counts().head(15))


## 13. Method summary

- **Base model:** Qwen3-VL-8B-Instruct with NF4 quantization and QLoRA adapters.
- **Prediction:** constrained scoring over task-specific labels.
- **Image views:** original, contrast-enhanced, and whitespace-cropped.
- **Aggregation:** combination of class log-probabilities across views.
- **Adaptive computation:** additional forward passes are triggered by low prediction margins.
- **Model selection:** calibration/holdout evaluation determines whether adaptive inference is used on the test set.
- **Evaluation:** comparisons of single-view, two-view, three-view, and adaptive policies, including average number of views processed.
- **Output:** `index,answer` CSV with the expected label format.

The multiview component increases inference-time computation without updating model parameters; QLoRA is applied during supervised fine-tuning.
